In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder, KBinsDiscretizer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import seaborn as sns  
import matplotlib.pyplot as plt


df_train = pd.read_csv('ecommerce_data_train.csv')
df_test = pd.read_csv('ecommerce_data_val.csv')

# # drop the rows with missing values
# df_train = df_train.dropna()
# df_test = df_test.dropna()

In [2]:
# Splitting the data into features and target variable
X_train = df_train.drop('Monthly Revenue', axis = 1)
y_train = df_train['Monthly Revenue']
# X_train = X_train.dropna()
# y_train = y_train.dropna()


X_test = df_test.drop('Monthly Revenue', axis = 1)
y_test = df_test['Monthly Revenue']
# X_test = X_test.dropna()
# y_test = y_test.dropna()


1) Start by training a model using only "Monthly Ad Spend" as a feature. This model only has two parameters to learn: an intercept (bias) term and a weight for Monthly Ad Spend. Train this model and report the RMSE (root mean squared error) for the test data (to the nearest integer)

In [3]:
# Question 1
# Train a model using only "Monthly Ad Spend" as a feature. 

Xq1_train = X_train.copy()
Xq1_test = X_test.copy()

theVars = ['Monthly Ad Spend']

Xq1_train = Xq1_train[theVars]
Xq1_test = Xq1_test[theVars]

# Now train with (y_train,Xq1_train)
# YOUR CODE HERE...
#
# 
# Train the model
model = LinearRegression()
model.fit(Xq1_train, y_train)
# Predictions
y_pred_train = model.predict(Xq1_train)
y_pred_test = model.predict(Xq1_test)

# Evaluate the model
# train_rmse = root_mean_squared_error(y_train, y_pred_train)
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(test_rmse.round(0))

21792.0


2) In this question you are asked to calculate the predicted test cases manually (rather than using my_model.predict). Suppose you named the model you trained in question 1 as my_model. You can then obtain the bias/intercept and slope term as 

bias = my_model.intercept_   and weight = my_model.coef_

Using this information, calculate the predicted values for the test data and then calculate the RMSE for the test data - to the nearest integer (Hint: It should equal what you got for  question 1) 

In [4]:
bias = model.intercept_
weight = model.coef_

# Manually calculate the predicted values for the test data
y_pred_manual = Xq1_test['Monthly Ad Spend'] * weight + bias
# Calculate the RMSE for the test data
manual_rmse = root_mean_squared_error(y_test, y_pred_manual)
print(manual_rmse.round(0))

21792.0


3) Now train a model predicting monthly revenue using all features in a linear model EXCEPT "Season". So you should drop the "Season" column from the feature array and then use all other columns as features in a linear model (check the associated notebook for help). What is the RMSE for this model (to the nearest integer) 



In [5]:
# Question 3
#
# Train a model using all features except for season - use all features linearly
#
Xq3_train = X_train.copy()
Xq3_test = X_test.copy()
Xq3_train = Xq3_train.drop('Season', axis = 1)   # this drops Season from the feature array
Xq3_test = Xq3_test.drop('Season', axis = 1)
# Now train with (y_train,Xq1_train)

theVars = ['Monthly Ad Spend', 'Number of Website Visits', 'Average Customer Rating', 'Competition', 'Income', 'Special Events']


# Xq3_train = Xq3_train[theVars]
# Xq3_test = Xq3_test[theVars]


# Now train with (y_train,Xq1_train)
# YOUR CODE HERE...
# 
# Train the model
model2 = LinearRegression()
model2.fit(Xq3_train, y_train)
# Predictions
y_pred_train = model2.predict(Xq3_train)
y_pred_test = model2.predict(Xq3_test)

# Evaluate the model
# train_rmse = root_mean_squared_error(y_train, y_pred_train)
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(test_rmse.round(0))


10955.0


4) Now add "Season" to the model as a one-hot coded feature along with the remaining numerical features. Do this new model lead to an improvement in predicting the test data?

In [6]:
# Question 4
#
# Now add season to the model as a onehot coded feature - do you do better on test data?
#
#
Xq4_train = X_train.copy()
Xq4_test = X_test.copy()

one_hot_encoder = ColumnTransformer(transformers=[('encoder', OneHotEncoder(drop='first'), ['Season'])],
                                     remainder='passthrough')
Xq4_train_encoded = one_hot_encoder.fit_transform(Xq4_train)
Xq4_test_encoded = one_hot_encoder.transform(Xq4_test)

# YOUR CODE HERE...
# 

model3 = LinearRegression()
model3.fit(Xq4_train_encoded, y_train)
# Predictions
y_pred_train = model3.predict(Xq4_train_encoded)
y_pred_test = model3.predict(Xq4_test_encoded)

# Evaluate the model
# train_rmse = root_mean_squared_error(y_train, y_pred_train)
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(test_rmse.round(0))

11073.0


5. You drop "Season" again and return to the original feature array in question 3. Compare the rmse on the test data in question 3 to one where you transform the "Number of Website Visits" feature into a onehot coded, discretized version of itself (see the jupyter notebook for details). Do you do better with the onehot coded version? 



In [7]:
# Question 5: Two models with 'Number of Website Visits' as numerical and categorical

# Numerical model
#
# this is just the same as in question 3
# rmse for 3 was 10955.0

# Categorical model (using KBinsDiscretizer for binning)
Xq5_train = Xq3_train.copy()
Xq5_test = Xq3_test.copy()


# Pipeline for discretizing and then one-hot encoding
pipe = Pipeline([
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None)),
    ('onehot', OneHotEncoder(drop='first'))
])

# ColumnTransformer
transformer = ColumnTransformer(
    transformers=[
        ('discretize_and_encode', pipe, ['Number of Website Visits'])
    ],
    remainder='passthrough'
)

Xq5_train_encoded = transformer.fit_transform(Xq5_train)
Xq5_test_encoded = transformer.transform(Xq5_test)

model3.fit(Xq3_train, y_train)
y_pred_test_q3 = model3.predict(Xq3_test)
test_rmse_q3 = root_mean_squared_error(y_test, y_pred_test_q3)


# YOUR CODE HERE...
# 
model5 = LinearRegression()
model5.fit(Xq5_train_encoded, y_train)

# Predictions
# y_pred_train = model5.predict(Xq5_train_encoded)
y_pred_test = model5.predict(Xq5_test_encoded)

# Evaluate the model
# train_rmse = root_mean_squared_error(y_train, y_pred_train)
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(test_rmse.round(0))
print(test_rmse_q3.round(0))

11072.0
10955.0


6) You return to the original feature array in question 3. However, now you decide to discretize and onehot code the "Income" feature using 5 bins (see the notebook for details). Does this new model have better predictive performance than the original? 



In [8]:
# Question 6: Try to discretize Income


# Categorical model (using KBinsDiscretizer for binning)
Xq6_train = Xq3_train.copy()
Xq6_test = Xq3_test.copy()


# Pipeline for discretizing and one-hot encoding "Income"
income_pipeline = Pipeline([
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')),
    ('onehot', OneHotEncoder(drop='first'))
])

# ColumnTransformer for applying the transformation only to "Income"
transformer_income = ColumnTransformer(
    transformers=[
        ('discretize_and_encode_income', income_pipeline, ['Income'])
    ],
    remainder='passthrough'
)

# YOUR CODE HERE...
# 
# Transform the training and test datasets
X_train_income_encoded = transformer_income.fit_transform(Xq6_train)
X_test_income_encoded = transformer_income.transform(Xq6_test)

# Train the model with the transformed data
model_income = LinearRegression()
model_income.fit(X_train_income_encoded, y_train)

# Predictions
y_pred_test_income = model_income.predict(X_test_income_encoded)

# Calculate RMSE
test_rmse_income = np.sqrt(mean_squared_error(y_test, y_pred_test_income))

print(test_rmse_income.round(0))
print(test_rmse_q3.round(0))

11528.0
10955.0


7. You return to the original feature array in question 3. However, now you decide to add another non-linearity: an interaction between "Income" and "Competition". You do this by simply creating a new column in the feature array which is the product of "Income" and "Competition" (see notebook). Train the model with this new feature added. Does this model have better predictive performance on the test data than the original one in question 3? 



In [9]:
# Question 7

Xq7_train = Xq3_train.copy()
Xq7_test = Xq3_test.copy()

# create interaction
Xq7_train['Income_Comp'] = Xq7_train['Income']*Xq7_train['Competition']
Xq7_test['Income_Comp'] = Xq7_test['Income']*Xq7_test['Competition']


# YOUR CODE HERE...
# 
model7 = LinearRegression()
model7.fit(Xq7_train, y_train)

# Predictions
y_pred_test_interaction = model7.predict(Xq7_test)

# Calculate RMSE for the new model
test_rmse_interaction = np.sqrt(mean_squared_error(y_test, y_pred_test_interaction))

print(test_rmse_interaction.round(0))
print(test_rmse_q3.round(0))

10082.0
10955.0
